<a href="https://colab.research.google.com/github/Fatima-Eman-hub/fatima-eman-flyrank-ml-01/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Lane 2 — Refresh / Content Opportunity Scoring**

**Task type: Scoring / Ranking**, built on top of a binary classification sub-problem.

The end output is a ranked queue — every page gets a score, and pages are sorted high to low so a reviewer works from the top — which makes this fundamentally a scoring/ranking task. Under the hood, the score comes from a classifier's predicted probability (is this page declining?) combined with a transparent baseline score, the same pattern the starter pipeline uses (`03_train_model.py` → `04_evaluate_and_export.py`). It is not clustering, since I am not grouping unlabeled pages into archetypes — I have a specific outcome I care about (decline). It is not pure classification either, because the product a reviewer actually sees is an ordered list under limited review capacity, not just a yes/no per page.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy label:** `is_declining_label = (trend_direction == "down")`

This label comes from a defined rule applied to a current-window derived bucket (`trend_direction`), not from a true future observed outcome — it is the beginner proxy the starter pipeline uses. Per the lane guide, a stronger capstone should move toward a future-window label instead, such as: features from the prior 90 days predicting decline over the next 30 days. I am using the starter proxy for this framing exercise since it lets me back my choice with the documented starter results, and I will revisit the label shape before Week 4 when I confirm or change my lane.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50**

A content team can only realistically review a limited number of pages, so the metric that matches the real decision is precision at the top of the ranked list, not overall accuracy (which would reward the model for being right about the many pages nobody is going to review anyway). Precision@50 asks: of the top 50 pages the system flags, how many are actually declining? The documented starter result gives a concrete bar: baseline rules score 0.240 at Precision@50, while a random forest scores 0.740 on the same task shape — meaning roughly 12 of the top 50 flagged pages are correct under the baseline versus roughly 37 under the model.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page. Build the target/proxy column right next to the
# observable signals that would feed a model.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

unit_of_analysis_cols = ["content_id", "impressions_90d", "days_since_last_update",
                          "avg_position", "ctr", "trend_direction", "is_declining_label"]

print(f"{len(df)} rows -> {len(df)} content pages, one row per page")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")
df[unit_of_analysis_cols].head(8)

30000 rows -> 30000 content pages, one row per page
Declining rate: 54.2%


,content_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,1724,22,21.2,0.06,stable,0


**Unit of analysis:**

one row = one content page. Confirmed above — each row has a unique `content_id`, and the target column `is_declining_label` sits right next to the observable signals (impressions, staleness, position, CTR) that would feed a model.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like `stale AND visible` (`days_since_last_update >= 180` and `impressions_90d >= 500`) can only combine a couple of signals with a hard cutoff — it can't weigh position, CTR, word count, and freshness together, and it treats every page that crosses the threshold identically regardless of degree. The starter results are direct evidence this matters: the hand rule scores Precision@50 = 0.240, while a random forest using the same observable signals scores 0.740 — roughly three times more of the top 50 flagged pages are actually correct. Critically, Notebook 02 showed this gain doesn't require a black box: a depth-3 decision tree, fully readable as if/else logic, already beats the hand rule at the same metric. So this is genuinely an ML problem, not a "just needs a smarter rule" problem — the signals interact in ways a single threshold can't capture, but the result stays explainable enough for a reviewer to trust and inspect.

In [2]:
# Show the documented comparison directly, so the claim above is backed by numbers,
# not just words.
print("Documented starter pipeline result (docs/ml-intern-dataset-and-lane-guide.md):")
print(f"{'Method':<22}{'ROC AUC':>10}{'Avg Precision':>16}{'Precision@50':>15}")
print(f"{'baseline rules':<22}{0.627:>10}{0.468:>16}{0.240:>15}")
print(f"{'logistic regression':<22}{0.700:>10}{0.522:>16}{0.400:>15}")
print(f"{'decision tree':<22}{0.742:>10}{0.575:>16}{0.540:>15}")
print(f"{'random forest':<22}{0.750:>10}{0.618:>16}{0.740:>15}")

gap = 0.740 - 0.240
print(f"\nPrecision@50 gap (random forest - baseline): {gap:.3f}"
      f" -> about {round(gap*50)} more correct pages in the top 50, out of 50.")

Documented starter pipeline result (docs/ml-intern-dataset-and-lane-guide.md):
Method                   ROC AUC   Avg Precision   Precision@50
baseline rules             0.627           0.468           0.24
logistic regression          0.7           0.522            0.4
decision tree              0.742           0.575           0.54
random forest               0.75           0.618           0.74

Precision@50 gap (random forest - baseline): 0.500 -> about 25 more correct pages in the top 50, out of 50.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.